# Phase 1 · Binary Classifier — Healthy vs Infected

**Standalone notebook.** Trains a Healthy/Infected classifier on the NIH malaria
single-cell dataset using transfer learning (ResNet50). At the end it saves the
trained model to `./artifacts/binary_resnet50.pt`, which later phases reuse to
decide *which cells are infected and therefore continue down the pipeline*.

### Pipeline context
**[Phase 1: Healthy/Infected]** → Phase 2: locate parasite → Phase 3: crop → Phase 4: stage

### Objectives
1. Load the NIH dataset (`Parasitized/`, `Uninfected/`).
2. Build ResNet50 (ImageNet-pretrained) with a 2-class head.
3. Train (freeze → fine-tune), evaluate, save the model.

## 1. Imports

In [1]:
import os, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report,
                             roc_curve, auc)
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


## 2. Configuration — set your dataset path here
`NIH_DATA_DIR` must contain `Parasitized/` and `Uninfected/` subfolders.

In [2]:
import kagglehub

NIH_DATA_DIR = kagglehub.dataset_download("iarunava/cell-images-for-detecting-malaria")
NIH_DATA_DIR = os.path.join(NIH_DATA_DIR, "cell_images")
# ✅ Handle double-nested cell_images
if os.path.isdir(os.path.join(NIH_DATA_DIR, "cell_images")):
    NIH_DATA_DIR = os.path.join(NIH_DATA_DIR, "cell_images")

print("Using dataset:", NIH_DATA_DIR)
print("Contents:", os.listdir(NIH_DATA_DIR))  # must show ONLY Parasitized, Uninfected

IMG_SIZE        = 224
BATCH_SIZE      = 32
SEED            = 42
EPOCHS_HEAD     = 5
EPOCHS_FINETUNE = 5
LR_HEAD, LR_FT  = 1e-3, 1e-4

ARTIFACTS = "./artifacts"; os.makedirs(ARTIFACTS, exist_ok=True)
CLASSES = ["Healthy", "Infected"]

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
assert os.path.isdir(NIH_DATA_DIR), f"Set NIH_DATA_DIR. Not found: {NIH_DATA_DIR}"

Using Colab cache for faster access to the 'cell-images-for-detecting-malaria' dataset.
Using dataset: /kaggle/input/cell-images-for-detecting-malaria/cell_images/cell_images
Contents: ['Uninfected', 'Parasitized']
Device: cpu


## 3. Datasets & dataloaders
ImageFolder + a stratified train/val/test split (NIH folders are read alphabetically: `Parasitized`=0, `Uninfected`=1).

In [3]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

base = datasets.ImageFolder(NIH_DATA_DIR)            # raw PIL, transform applied below
print("Folder -> index:", base.class_to_idx)
targets = [t for _, t in base.samples]

idx = np.arange(len(targets))
tv, test_idx = train_test_split(idx, test_size=0.15, stratify=targets, random_state=SEED)
tr_idx, val_idx = train_test_split(tv, test_size=0.1765,
                                   stratify=np.array(targets)[tv], random_state=SEED)

class DS(torch.utils.data.Dataset):
    def __init__(self, base, indices, tf): self.b, self.ix, self.tf = base, indices, tf
    def __len__(self): return len(self.ix)
    def __getitem__(self, i):
        p, y = self.b.samples[self.ix[i]]
        return self.tf(self.b.loader(p)), y

train_loader = DataLoader(DS(base, tr_idx,  train_tf), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(DS(base, val_idx, eval_tf),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(DS(base, test_idx,eval_tf),  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"train {len(tr_idx)} | val {len(val_idx)} | test {len(test_idx)}")

Folder -> index: {'Parasitized': 0, 'Uninfected': 1}
train 19289 | val 4135 | test 4134


## 4. Model — ResNet50 transfer learning
Replace only the final FC layer; freeze the backbone for the first phase of training.

In [4]:
def build_model(freeze=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, len(CLASSES))   # trainable head
    return m

model = build_model(freeze=True).to(device)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 118MB/s]


Trainable params: 4098


## 5. Training utilities

In [5]:
def run_epoch(loader, train, criterion, optimizer=None):
    model.train() if train else model.eval()
    tot_loss = correct = total = 0
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            if train: optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            if train: loss.backward(); optimizer.step()
            tot_loss += loss.item()*x.size(0)
            correct  += (out.argmax(1)==y).sum().item(); total += x.size(0)
    return tot_loss/total, correct/total

def fit(epochs, lr, history, ckpt):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad],
                                 lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", factor=0.2, patience=2)
    best = min(history["val_loss"]) if history["val_loss"] else np.inf
    for e in range(epochs):
        tl, ta = run_epoch(train_loader, True,  criterion, optimizer)
        vl, va = run_epoch(val_loader,   False, criterion)
        sched.step(vl)
        for k, v in zip(history, [tl, vl, ta, va]): history[k].append(v)
        print(f"epoch {len(history['train_loss']):02d} | train {tl:.4f}/{ta:.4f} | val {vl:.4f}/{va:.4f}")
        if vl < best:
            best = vl; torch.save({"model_state": model.state_dict(), "classes": CLASSES}, ckpt)
            print("  saved best ->", ckpt)
    return history

## 6. Train — head only, then fine-tune

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
CKPT = os.path.join(ARTIFACTS, "binary_resnet50.pt")

history = fit(EPOCHS_HEAD, LR_HEAD, history, CKPT)

for p in model.parameters(): p.requires_grad = True   # unfreeze
history = fit(EPOCHS_FINETUNE, LR_FT, history, CKPT)

  0%|          | 0/603 [00:00<?, ?it/s]

## 7. Training curves

In [ ]:
ep = range(1, len(history["train_loss"])+1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, history["train_loss"], "o-", label="train"); ax[0].plot(ep, history["val_loss"], "s-", label="val")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, history["train_acc"], "o-", label="train"); ax[1].plot(ep, history["val_acc"], "s-", label="val")
ax[1].set_title("Accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Evaluate on the test set
Accuracy, precision/recall/F1, confusion matrix, classification report and ROC/AUC.

In [ ]:
ckpt = torch.load(CKPT, map_location=device); model.load_state_dict(ckpt["model_state"]); model.eval()
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in tqdm(test_loader, leave=False):
        p = torch.softmax(model(x.to(device)), 1).cpu().numpy()
        y_prob.append(p); y_pred.append(p.argmax(1)); y_true.append(y.numpy())
y_true = np.concatenate(y_true); y_pred = np.concatenate(y_pred); y_prob = np.concatenate(y_prob)
print("")
# Map folder indices to friendly names for the report.
idx_name = {v: ("Healthy" if k.lower().startswith("uninfect") else "Infected")
            for k, v in base.class_to_idx.items()}
names = [idx_name[i] for i in range(len(idx_name))]
pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary")
print(f"accuracy {accuracy_score(y_true, y_pred):.4f} | precision {pr:.4f} | recall {rc:.4f} | f1 {f1:.4f}")
print(classification_report(y_true, y_pred, target_names=names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names); ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix")
for i in range(len(names)):
    for j in range(len(names)): ax.text(j, i, cm[i, j], ha="center", va="center")
plt.tight_layout(); plt.show()

infected = [i for i, n in enumerate(names) if n == "Infected"][0]
fpr, tpr, _ = roc_curve((y_true == infected).astype(int), y_prob[:, infected])
plt.figure(figsize=(4.5, 4)); plt.plot(fpr, tpr, label=f"AUC={auc(fpr,tpr):.4f}")
plt.plot([0,1],[0,1],"k--",alpha=.5); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC curve"); plt.legend(); plt.tight_layout(); plt.show()

## 9. Conclusion & handoff to Phase 2
* The trained binary model is saved at **`./artifacts/binary_resnet50.pt`**.
* **Phase 2** loads infected cells (those this model labels *Infected*) and runs a
  YOLOv8 detector to locate the parasite inside them.